In [ ]:
import baostock as bs
import pandas as pd
from datetime import datetime, timedelta
import time

def get_target_stocks(target_date=None):
    """
    通过 Baostock 获取全市场 A 股数据，筛选市值最小的前50只股票。
    
    Args:
        target_date: 查询日期 (YYYY-MM-DD)，默认为今天。如果是周末/节假日会自动回退到最近交易日。
    """
    if target_date is None:
        target_date = datetime.today().strftime('%Y-%m-%d')

    print(f"正在登录 Baostock 并获取 {target_date} 的数据...")

    # 带重试的登录
    max_login_retries = 3
    for attempt in range(max_login_retries):
        try:
            lg = bs.login()
            if lg.error_code == '0':
                print(f"登录成功！")
                break
            else:
                print(f"登录失败 (attempt {attempt+1}): error_code={lg.error_code}, msg={lg.error_msg}")
                if attempt < max_login_retries - 1:
                    time.sleep(3)
        except Exception as e:
            print(f"登录异常 (attempt {attempt+1}): {e}")
            if attempt < max_login_retries - 1:
                time.sleep(3)
    else:
        print("❌ Baostock 登录连续失败，请检查网络连接。")
        print("提示：Baostock 服务器偶尔不稳定，请稍后重试。")
        return None

    # 1. 获取当天所有的股票代码
    # 如果指定日期无数据（非交易日），自动尝试前1-5天
    stock_df = pd.DataFrame()
    for back in range(6):  # 最多回退6天
        try_date = (datetime.strptime(target_date, '%Y-%m-%d') - timedelta(days=back)).strftime('%Y-%m-%d')
        rs_all = bs.query_all_stock(day=try_date)
        if rs_all.error_code == '0':
            stock_df = rs_all.get_data()
            if not stock_df.empty:
                print(f"获取到 {try_date} 的股票列表，共 {len(stock_df)} 只")
                target_date = try_date  # 更新为实际使用的日期
                break
        print(f"{try_date} 无数据 (error: {rs_all.error_msg})，尝试前一天...")

    if stock_df.empty:
        print("❌ 未获取到股票列表，Baostock 服务器可能不可用。")
        print("建议：1) 检查网络 2) 稍后重试 3) 或改用 akshare_choose.ipynb 作为替代方案")
        bs.logout()
        return None

    data_list = []
    total_stocks = len(stock_df)

    print(f"共获取到 {total_stocks} 只股票，开始逐个拉取数据（此过程可能需要几分钟，请耐心等待）...")
    print("提示：如需中止请按 Ctrl+C")

    # 2. 遍历所有股票，获取所需的 K线和估值数据
    processed = 0
    for index, row in stock_df.iterrows():
        code = row['code']
        # 排除北交所或无效代码，仅保留沪深A股 (sh.6, sz.0, sz.3 开头)
        if not (code.startswith('sh.6') or code.startswith('sz.0') or code.startswith('sz.3')):
            continue

        try:
            # 获取数据：收盘价，交易状态，是否ST，滚动市盈率，市净率，成交量(股)，换手率(%)
            rs_data = bs.query_history_k_data_plus(
                code,
                "code,close,tradestatus,isST,peTTM,pbMRQ,volume,turn",
                start_date=target_date, end_date=target_date,
                frequency="d", adjustflag="3"
            )

            if rs_data.error_code == '0' and len(rs_data.data) > 0:
                row_data = rs_data.get_data().iloc[0]

                # 数据清洗与类型转换
                try:
                    close = float(row_data['close'])
                    tradestatus = str(row_data['tradestatus'])
                    isST = str(row_data['isST'])
                    pe = float(row_data['peTTM']) if row_data['peTTM'] else -1.0
                    pb = float(row_data['pbMRQ']) if row_data['pbMRQ'] else -1.0
                    volume = float(row_data['volume']) if row_data['volume'] else 0.0
                    turn = float(row_data['turn']) if row_data['turn'] else 0.0
                except (ValueError, TypeError):
                    continue

                # 3. 核心条件过滤
                # 非停牌 (tradestatus == '1')
                # 非ST (isST == '0')
                # PE > 0 且 PB > 0
                if tradestatus == '1' and isST == '0' and pe > 0 and pb > 0:
                    # 4. 计算流通市值 (换手率单位是%，所以需要除以100)
                    if turn > 0:
                        # 流通股本 = volume / (turn / 100)
                        # 流通市值 = close * 流通股本
                        market_cap = close * (volume / (turn / 100))

                        data_list.append({
                            'code': code,
                            'close': close,
                            'pe': pe,
                            'pb': pb,
                            'market_cap': market_cap
                        })
        except Exception as e:
            # 单只股票出错不影响整体流程
            pass

        processed += 1
        if processed % 500 == 0:
            print(f"  进度: {processed}/{total_stocks}...")

    bs.logout()
    print(f"数据抓取完成（处理 {processed} 只），正在进行排序过滤...")

    # 将结果转换为 DataFrame
    result_df = pd.DataFrame(data_list)

    if result_df.empty:
        print("没有符合条件的股票。")
        return result_df

    # 5. 按市值从小到大排序，并截取前 50 只
    top_50_smallest_cap = result_df.sort_values(by='market_cap', ascending=True).head(50)

    # 6. 在这 50 只股票的基础上，按价格由低到高排序
    final_result = top_50_smallest_cap.sort_values(by='close', ascending=True)

    # 重置索引
    final_result = final_result.reset_index(drop=True)

    return final_result

# 运行代码
if __name__ == '__main__':
    # 建议填入最近的一个交易日，例如 '2026-04-01'
    # 如果不填日期，默认使用今天。若今天是周末/节假日，函数会自动回退到最近的交易日。
    df = get_target_stocks('2026-04-01')

    if df is not None and not df.empty:
        print("\n=== 最终筛选结果（市值最小的前50只，并按价格由低到高排序） ===")
        # 打印时调整市值的显示格式（转换为"亿元"单位方便阅读）
        df = df.copy()
        df['market_cap_亿元'] = df['market_cap'] / 100000000
        pd.set_option('display.max_rows', 50)
        print(df[['code', 'close', 'pe', 'pb', 'market_cap_亿元']])
    else:
        print("\n未获取到有效数据。")
        print("可能原因：1) Baostock 服务器不可用 2) 指定日期非交易日 3) 网络问题")
        print("建议：改用 akshare_choose.ipynb 作为替代方案（使用东方财富数据源）")
